# Context Engineering Playground

> *"Prompt engineering is about crafting a single instruction. Context engineering is about filling the context window with just the right information for the next step."*  
> — Andrej Karpathy

Hands-on companion to the **Context Engineering Cheatsheet**. Each section pairs a concept with runnable experiments so you can observe failure modes directly.

**Sections:** Setup → Context Rot → Failure Mode Triggers → The Four Operations → RAG Pattern → Multi-Agent Isolation → Compression

**Prerequisites:** `anthropic` Python SDK, `ANTHROPIC_API_KEY` in environment.

---
## 1. Setup

In [1]:
%pip install anthropic numpy --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, textwrap, random
from dataclasses import dataclass, field
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY', 'YOUR_KEY_HERE'))
MODEL = 'claude-haiku-4-5-20251001'   # fast + cheap for experiments

def chat(messages, system='', max_tokens=512):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages)
    if system:
        kwargs['system'] = system
    return client.messages.create(**kwargs).content[0].text

def token_count(text): return len(str(text)) // 4
def wrap(text, width=80): return textwrap.fill(text, width)

print('Setup complete:', MODEL)

Setup complete: claude-haiku-4-5-20251001


---
## 2. Context Rot Demo

A fact is buried in the context window. As we add filler text around it, retrieval accuracy drops — even though the answer is still present.

This mirrors Chroma's finding: *coherent haystacks can perform **worse** than shuffled text, and accuracy degrades unpredictably.*

In [ ]:
NEEDLE = 'The internal project codename for the 2031 budget cycle is GOLDFISH.'
QUESTION = 'What is the internal project codename for the 2031 budget cycle?'
FILLER = 'The quarterly infrastructure review encompasses a comprehensive assessment of all cloud-native workloads running across the production and staging environments. Capacity utilization metrics are aggregated from vSphere telemetry and cross-referenced against projected demand curves supplied by the business unit leads. Any anomalies exceeding a two-sigma deviation trigger an automated escalation to the on-call platform engineer, who has a fifteen-minute SLA for initial triage.'

def build_haystack(needle, filler, copies, position='middle'):
    block = ('\n\n' + filler) * copies
    half = len(block) // 2
    if position == 'start': return needle + block
    if position == 'end':   return block + '\n\n' + needle
    return block[:half] + '\n\n' + needle + block[half:]

def run_niah(copies_list, position='middle', trials=3):
    print(f"{'Copies':>8}  {'~Tokens':>8}  {'Correct':>8}")
    print('-' * 40)
    for copies in copies_list:
        haystack = build_haystack(NEEDLE, FILLER, copies, position)
        correct = 0
        for _ in range(trials):
            prompt = f'Context:\n{haystack}\n\nQuestion: {QUESTION}\nAnswer in one sentence.'
            ans = chat([{'role': 'user', 'content': prompt}], max_tokens=60)
            if 'GOLDFISH' in ans.upper(): correct += 1
            time.sleep(0.3)
        print(f'{copies:>8}  {token_count(haystack):>8,}  {correct}/{trials}')

run_niah([0, 5, 15, 30, 60], position='middle', trials=3)

In [ ]:
# Exercise 2-A: Position bias — start vs middle vs end
for pos in ['start', 'middle', 'end']:
    print(f'\n── Position: {pos.upper()} ──')
    run_niah([20], position=pos, trials=5)

In [ ]:
# Exercise 2-B: Coherent vs shuffled haystack
def shuffled(text):
    w = text.split(); random.shuffle(w); return ' '.join(w)

for label, filler in [('Coherent', FILLER), ('Shuffled', shuffled(FILLER))]:
    correct = 0
    for _ in range(5):
        hs = build_haystack(NEEDLE, filler, 30, 'middle')
        ans = chat([{'role': 'user', 'content': f'Context:\n{hs}\n\nQuestion: {QUESTION}\nAnswer in one sentence.'}], max_tokens=60)
        if 'GOLDFISH' in ans.upper(): correct += 1
        time.sleep(0.3)
    print(f'{label:>10}: {correct}/5')

---
## 3. Failure Mode Triggers

Each cell deliberately induces a failure mode, then demonstrates a mitigation.

In [ ]:
# 3-A: Context Poisoning — hallucination in turn N feeds back as fact in turn N+1
print('FAILURE: Context Poisoning\n')

q1 = "What specific port does the fictional AcmeCorp tool 'DataRouter v3' use?"
t1 = chat([{'role': 'user', 'content': q1}], max_tokens=80)
print('Turn 1 (hallucination risk):', wrap(t1))

# Inject the (possibly fabricated) answer back as established fact
msgs = [
    {'role': 'user',      'content': q1},
    {'role': 'assistant', 'content': t1},
    {'role': 'user',      'content': 'Given that port, what firewall rule should we add?'},
]
t2 = chat(msgs, max_tokens=120)
print('\nTurn 2 (poison propagated):', wrap(t2))
print('\n^ The fabricated port is now ground truth.')

In [ ]:
# 3-A Mitigation: tag uncertain answers before injecting them
tagged = f'[UNVERIFIED — may be hallucinated]\n{t1}'
msgs_safe = [
    {'role': 'user',      'content': q1},
    {'role': 'assistant', 'content': tagged},
    {'role': 'user',      'content': 'Given that port, what firewall rule should we add?'},
]
safe = chat(msgs_safe, system='When context is [UNVERIFIED], caveat all downstream recommendations.', max_tokens=150)
print('Mitigation result:', wrap(safe))

In [ ]:
# 3-B: Context Clash — contradictory sources, which wins?
print('FAILURE: Context Clash\n')

clash_prompt = """Document A (wiki, 2024-01-15): The deployment timeout is 300 seconds.
Document B (runbook, 2024-11-02): The deployment timeout is 120 seconds.
Document C (Slack, 2025-02-10): We bumped the timeout to 600s last week.

What is the current deployment timeout?"""

print('Without guidance:')
print(wrap(chat([{'role': 'user', 'content': clash_prompt}], max_tokens=150)))

print('\nWith conflict-resolution rule:')
print(wrap(chat([{'role': 'user', 'content': clash_prompt}],
                system='When sources conflict, prefer the most recent by date. State which source you chose.',
                max_tokens=150)))

In [ ]:
# 3-C: Entity Resolution Failure — two things share a name
print('FAILURE: Entity Resolution\n')

ep = """Project Alpha [infra] started in 2022, budget $2.4M.
Project Alpha [mobile app] launched 2024, 12 engineers.

How many engineers are on Project Alpha?"""

print('Ambiguous:', wrap(chat([{'role': 'user', 'content': ep}], max_tokens=80)))

ep2 = ep.replace('Project Alpha [infra]', 'INFRA-001').replace('Project Alpha [mobile app]', 'MOBILE-002').replace('Project Alpha?', 'MOBILE-002?')
print('\nWith IDs:', wrap(chat([{'role': 'user', 'content': ep2}], max_tokens=80)))

---
## 4. The Four Operations

LangChain's framework: **Write** / **Select** / **Compress** / **Isolate**

In [ ]:
# WRITE: external memory store — persist info outside the context window
print('OPERATION: WRITE\n')

class Scratchpad:
    def __init__(self): self._store = {}
    def save(self, k, v): self._store[k] = v; print(f'  [WRITE] {k} → {v[:60]}')
    def load(self, k):    v = self._store.get(k); print(f'  [READ]  {k} → {v or "NOT FOUND"[:60]}'); return v

mem = Scratchpad()

# Session 1: learn preference
chat([{'role': 'user', 'content': 'My name is Alex and I prefer bullet-point responses.'}], max_tokens=50)
mem.save('user.name', 'Alex')
mem.save('user.format', 'bullet points')

# Session 2: new empty context, reload from store
name   = mem.load('user.name')
fmt    = mem.load('user.format')
system = f"User's name is {name}. They prefer {fmt}."
resp   = chat([{'role': 'user', 'content': 'What are the benefits of context engineering?'}], system=system, max_tokens=200)
print('\nNew-session response (memory injected):')
print(resp)

In [ ]:
# SELECT: pull only relevant chunks into context
print('OPERATION: SELECT\n')

KB = [
    {'id': 'k1', 'topic': 'deployment', 'text': 'Blue-green deployments use health checks and automatic rollback if error rate > 2% within 5 minutes.'},
    {'id': 'k2', 'topic': 'auth',       'text': 'OAuth 2.0 PKCE flow. Tokens expire in 1 hour, refresh tokens in 30 days.'},
    {'id': 'k3', 'topic': 'database',   'text': 'PostgreSQL 15, read replicas in us-east-2 and eu-west-1, connection pooling via PgBouncer.'},
    {'id': 'k4', 'topic': 'monitoring', 'text': 'Metrics to Wavefront, alerts to PagerDuty. SLO: 99.9% uptime, p99 < 200ms.'},
    {'id': 'k5', 'topic': 'security',   'text': 'All secrets in HashiCorp Vault. Production access requires MFA and manager approval.'},
]

def keyword_select(query, kb, top_k=2):
    qw = set(query.lower().split())
    scored = sorted(kb, key=lambda e: len(qw & set((e['topic']+' '+e['text']).lower().split())), reverse=True)
    return scored[:top_k]

q = 'How does the deployment rollback work?'
selected = keyword_select(q, KB, top_k=2)
print(f'Query: {q}')
print(f'Selected {len(selected)}/{len(KB)} chunks:')
for s in selected: print(f'  [{s["id"]}] {s["text"][:70]}')

ctx = '\n\n'.join(f'[{s["id"]}] {s["text"]}' for s in selected)
ans = chat([{'role': 'user', 'content': f'Context:\n{ctx}\n\nQuestion: {q}'}], max_tokens=120)
print('\nAnswer:', wrap(ans))

In [ ]:
# COMPRESS: summarize older turns to reclaim token budget
print('OPERATION: COMPRESS\n')

HISTORY = [
    {'role': 'user',      'content': 'Tell me about context engineering.'},
    {'role': 'assistant', 'content': 'Context engineering is the practice of managing what goes into the LLM context window across turns, tool calls, and sessions.'},
    {'role': 'user',      'content': 'What are the main failure modes?'},
    {'role': 'assistant', 'content': 'Context poisoning, distraction, clash, confusion, rot, entity resolution failures, and multi-agent incoherence.'},
    {'role': 'user',      'content': 'How does RAG help?'},
    {'role': 'assistant', 'content': 'RAG pulls only relevant docs into context, preventing distraction from unrelated material.'},
]

print(f'Before: {len(HISTORY)} messages, ~{token_count(HISTORY)} tokens')

to_compress = HISTORY[:-2]
recent      = HISTORY[-2:]
transcript  = '\n'.join(f'{m["role"].upper()}: {m["content"]}' for m in to_compress)
summary     = chat([{'role': 'user', 'content': f'Summarize in 2 sentences, retaining key facts:\n\n{transcript}'}], max_tokens=120)

print(f'After:  {len(recent)} recent + 1 summary')
print(f'Summary tokens: ~{token_count(summary)}')
print('Summary:', wrap(summary))

In [ ]:
# ISOLATE: split context across specialist agents, aggregate via coordinator
print('OPERATION: ISOLATE\n')

PROPOSAL = 'Migrate the auth service to a stateless JWT microservice in Go. Store refresh tokens in Redis. Complete in one sprint (2 weeks).'

critic_report   = chat([{'role': 'user', 'content': f'List the top 3 technical risks:\n{PROPOSAL}'}],
                        system='You are a rigorous technical critic. Focus only on technical risk.',
                        max_tokens=200)
advocate_report = chat([{'role': 'user', 'content': f'Describe the fastest path to ship this:\n{PROPOSAL}'}],
                        system='You are a pragmatic engineering lead. Focus only on velocity.',
                        max_tokens=200)

print('── Critic ──'); print(wrap(critic_report))
print('\n── Advocate ──'); print(wrap(advocate_report))

# Coordinator sees only the two reports — not the original proposal or agent histories
synthesis = chat(
    [{'role': 'user', 'content': f'Critic:\n{critic_report}\n\nAdvocate:\n{advocate_report}\n\nSynthesize into one balanced recommendation.'}],
    system='Synthesize only what you are given. Do not add new analysis.',
    max_tokens=250,
)
print('\n══ Coordinator ══'); print(wrap(synthesis))

---
## 5. RAG Pattern

Minimal retrieval-augmented generation: rank docs by relevance, inject top-K into context. Uses Claude for relevance scoring (swap with cosine similarity over embeddings for production).

In [ ]:
DOCS = [
    {'id': 'd1', 'text': 'Blue-green deployments route 100% of traffic to the new environment after health checks. Old environment stays live for instant rollback.'},
    {'id': 'd2', 'text': 'Canary releases incrementally shift traffic (1%, 5%, 25%, 100%) gated by error rate and latency SLOs.'},
    {'id': 'd3', 'text': 'Feature flags decouple deployment from release. Code ships dark; features activate per-user via LaunchDarkly or Unleash.'},
    {'id': 'd4', 'text': 'OAuth 2.0 with PKCE: authorization codes are single-use and bound to a code verifier to prevent interception.'},
    {'id': 'd5', 'text': 'JWT tokens: sign with RS256 or ES256. Include exp/iat/sub claims. Never store sensitive data in payload — it is trivially base64-decoded.'},
    {'id': 'd6', 'text': 'HashiCorp Vault dynamic secrets: database creds with TTL, AWS STS tokens scoped to a role, on-demand PKI certs.'},
    {'id': 'd7', 'text': 'PgBouncer transaction-mode pooling is incompatible with prepared statements. Switch to session-mode or use the $pgbouncer virtual database.'},
    {'id': 'd8', 'text': 'PostgreSQL VACUUM reclaims storage from dead tuples. Monitor pg_stat_user_tables.n_dead_tup for autovacuum lag on high-write tables.'},
]

def rank_docs(query, docs, top_k=2):
    doc_list = '\n'.join(f'[{d["id"]}] {d["text"]}' for d in docs)
    raw = chat([{'role': 'user', 'content': f'Rate each document 0-10 for relevance to: "{query}"\nReply ONLY with JSON mapping id to score.\n\nDocs:\n{doc_list}'}], max_tokens=150)
    cleaned = raw.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
    try:
        scores = json.loads(cleaned)
        return sorted(docs, key=lambda d: scores.get(d['id'], 0), reverse=True)[:top_k]
    except:
        return docs[:top_k]

def rag_answer(query, corpus, top_k=2):
    retrieved = rank_docs(query, corpus, top_k)
    print(f'Retrieved for: {query!r}')
    for r in retrieved: print(f'  [{r["id"]}] {r["text"][:65]}...')
    ctx = '\n\n'.join(f'[{r["id"]}] {r["text"]}' for r in retrieved)
    ans = chat(
        [{'role': 'user', 'content': f'Using only this context, answer: {query}\n\nContext:\n{ctx}'}],
        system='Answer concisely from context only. If context does not cover it, say so.',
        max_tokens=180,
    )
    return ans

QUERIES = [
    'What is the difference between blue-green and canary deployments?',
    'How should I store JWT tokens securely?',
    'Why is PgBouncer breaking my prepared statements?',
]

for q in QUERIES:
    print('\n' + '=' * 60)
    print('Answer:', wrap(rag_answer(q, DOCS)))
    time.sleep(0.5)

In [ ]:
# Exercise 5-A: Out-of-scope query — RAG should admit it, not hallucinate
q = "What is the company's parental leave policy?"
print(f'Out-of-scope: {q!r}\n')
print(wrap(rag_answer(q, DOCS)))

---
## 6. Multi-Agent Isolation

Context isolation prevents anchoring bias between agents with different roles. This experiment shows what happens when isolation breaks.

In [ ]:
@dataclass
class Agent:
    name: str
    system: str
    history: list = field(default_factory=list)

    def send(self, msg, max_tokens=250):
        self.history.append({'role': 'user', 'content': msg})
        reply = chat(self.history, system=self.system, max_tokens=max_tokens)
        self.history.append({'role': 'assistant', 'content': reply})
        return reply

critic   = Agent('Critic',   'Rigorous technical critic. Identify risks only. Ignore cost/business.')
advocate = Agent('Advocate', 'Pragmatic engineering lead. Fastest path to ship. Ignore edge cases.')

PROPOSAL = 'Migrate auth service to stateless JWT microservice in Go. Refresh tokens in Redis. Ship in 2 weeks.'

critic_out   = critic.send(f'Top 3 technical risks for: {PROPOSAL}')
advocate_out = advocate.send(f'Fastest path to ship: {PROPOSAL}')

print('── Isolated Critic ──'); print(wrap(critic_out))
print('\n── Isolated Advocate ──'); print(wrap(advocate_out))

In [ ]:
# Exercise 6-A: What if Critic sees Advocate's output first? (anchoring contamination)
contaminated = Agent('Contaminated Critic', critic.system)
contaminated_out = contaminated.send(
    f'Another engineer said: "{advocate_out}"\n\nNow give top 3 technical risks for: {PROPOSAL}'
)
print('── Contaminated Critic (saw Advocate first) ──')
print(wrap(contaminated_out))
print('\n^ Compare to isolated Critic above. Anchoring typically softens the critique.')

---
## 7. Compression / Summarization

Two strategies: **rolling summary** (compress as you go) and **checkpoint reset** (extract structured state, discard history).

In [ ]:
# 7-A: Rolling Summary Chat

class RollingChat:
    def __init__(self, system='', compress_every=4, keep_recent=2):
        self.system         = system
        self.compress_every = compress_every
        self.keep_recent    = keep_recent
        self.summary        = ''
        self.recent         = []
        self.compressions   = 0

    def _compress(self):
        if len(self.recent) < self.compress_every: return
        to_sum      = self.recent[:-self.keep_recent]
        self.recent = self.recent[-self.keep_recent:]
        transcript  = '\n'.join(f'{m["role"].upper()}: {m["content"]}' for m in to_sum)
        prefix      = f'Prior summary:\n{self.summary}\n\n' if self.summary else ''
        self.summary = chat([{'role': 'user', 'content': f'{prefix}Summarize in 2 sentences:\n{transcript}'}], max_tokens=120)
        self.compressions += 1

    def send(self, msg, max_tokens=180):
        self.recent.append({'role': 'user', 'content': msg})
        sys = (f'[Summary]\n{self.summary}\n\n' if self.summary else '') + self.system
        reply = chat(self.recent, system=sys, max_tokens=max_tokens)
        self.recent.append({'role': 'assistant', 'content': reply})
        self._compress()
        return reply

    def stats(self):
        return f'recent={len(self.recent)} msgs  compressions={self.compressions}  ~{token_count(self.summary)+token_count(self.recent)} tokens'

convo = RollingChat(system='You are a helpful platform engineering assistant.', compress_every=4, keep_recent=2)

questions = [
    'What is Tanzu Application Service?',
    'How does TAS differ from vanilla Kubernetes?',
    'What are the main TAS components?',
    'What is BOSH and why does TAS use it?',
    'How does Diego replace the old DEA architecture?',
    'What is CredHub used for in TAS?',
    'How do I scale a TAS foundation horizontally?',
]

for i, q in enumerate(questions):
    reply = convo.send(q)
    print(f'[{i+1}] Q: {q}')
    print(f'    A: {reply[:100].strip()}...')
    print(f'    {convo.stats()}\n')
    time.sleep(0.3)

if convo.summary:
    print('Final summary:', wrap(convo.summary))

In [ ]:
# 7-B: Checkpoint Reset — extract structured state, discard history, restart clean

def extract_state(history):
    transcript = '\n'.join(f'{m["role"].upper()}: {m["content"][:200]}' for m in history)
    raw = chat([{'role': 'user', 'content': f'Extract conversation state as JSON with fields: topic, decisions (list), open_questions (list), key_facts (dict). Reply ONLY with JSON.\n\n{transcript}'}], max_tokens=350)
    cleaned = raw.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
    try:    return json.loads(cleaned)
    except: return {'raw': raw}

state = extract_state(convo.recent)
print('Checkpoint state:')
print(json.dumps(state, indent=2))

# Fresh session — history gone, only structured state in system prompt
new_system = f'You are a platform engineering assistant.\n\nCheckpoint (prior history cleared):\n{json.dumps(state, indent=2)}'
resp = chat([{'role': 'user', 'content': 'What should I prioritize first for a new TAS foundation?'}], system=new_system, max_tokens=220)
print('\nPost-reset response:')
print(wrap(resp))
print(f'\nState tokens: ~{token_count(json.dumps(state))}  (vs full history: much more)')

---
## 8. Pattern → Failure Mode Map

| Pattern | Addresses | Section |
|---|---|---|
| Uncertainty tagging | Context Poisoning | 3-A |
| Selective retrieval (RAG) | Context Distraction, Confusion | 4-B, 5 |
| Conflict-resolution rules | Context Clash | 3-B |
| Disambiguating IDs | Entity Resolution Failure | 3-C |
| Rolling summary | Context Rot | 7-A |
| Checkpoint reset | Context Rot (severe) | 7-B |
| Agent isolation | Multi-Agent Incoherence | 4-D, 6 |
| External memory (Write op) | State persistence across sessions | 4-A |

## 9. Decision Guide

**Simple prompting is enough when:** single-turn task, < 10 turns / < 20K tokens, user provides all context.

**Invest in context engineering when:** agent loops with tool calls, multi-session continuity, knowledge base too large for one prompt, multiple agents must collaborate, you observe degradation or hallucination loops across turns.

---
*Sources: Karpathy (X, 2025) · LangChain Context Engineering for Agents · Chroma Research: Context Rot · Addy Osmani: Bringing Engineering Discipline to Prompts · Prompting Guide · Zep: What is Context Engineering?*